# Seq2Seq workshop — Sutskever et al. 2014 (`mid`)

Paper: [Sequence to Sequence Learning with Neural Networks](https://arxiv.org/abs/1409.3215)
([HTML](https://arxiv.org/html/1409.3215v3)). Code: `src/seq2seq/`. Preset: **`mid`** (one-GPU scale).

### Notation (used everywhere below)

| Symbol | Meaning | `mid` value |
|--------|---------|-------------|
| $B$ | batch size | 64 |
| $T$ | source length (after reverse + EOS) | $\le 51$ |
| $T'$ | target length (with SOS/EOS) | $\le 51$ |
| $L$ | LSTM depth (encoder and decoder) | 4 |
| $E$ | embedding dimension | 256 |
| $H$ | LSTM hidden / cell size | 256 |
| $\|V_x\|$, $\|V_y\|$ | source / target vocab sizes | $\le$ 20k each |
| $x_{1:T}$ | source token ids | shape $(B,T)$ |
| $y_{1:T'}$ | target token ids | shape $(B,T')$ |
| $e_t\in\mathbb{R}^{E}$ | embedding of one token | |
| $h^\ell_t\in\mathbb{R}^{H}$ | hidden state, layer $\ell$, time $t$ | |
| $c^\ell_t\in\mathbb{R}^{H}$ | cell state (LSTM only) | |
| $v$ | fixed sentence vector = encoder final state | $(L,B,H)$ for $(h,c)$ |

### Objective (§2)

$$
p(y_1,\ldots,y_{T'}\mid x)=\prod_{t=1}^{T'} p(y_t\mid v,\,y_{<t}),
\qquad v=q(x_1,\ldots,x_T).
$$

Encoder $q$ is a deep LSTM; decoder is an LM whose **initial** $(h,c)$ is $v$. Softmax over full $\|V_y\|$ each step.


## 0. Environment


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from seq2seq.config import mid_config, paper_config
from seq2seq.data import basic_tokenize, prepare_data, SPECIAL_TOKENS
from seq2seq.lstm_cell import LSTMCell
from seq2seq.deep_lstm import DeepLSTM
from seq2seq.model import Seq2Seq
from seq2seq.train import train, learning_rate_at_epoch
from seq2seq.decode import (
    load_checkpoint, greedy_decode, beam_search_decode, encode_source_sentence,
)
import torch

assert torch.cuda.is_available(), "Need CUDA for mid"
device = torch.device("cuda")
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

cfg = mid_config()
cfg.device = "cuda"
cfg.checkpoint_dir = str(ROOT / "runs" / "mid")
CKPT = Path(cfg.checkpoint_dir) / "checkpoint.pt"
HIST = Path(cfg.checkpoint_dir) / "monitor_history.json"
print("checkpoint:", CKPT, "exists=", CKPT.exists())


## 1. Specs — paper vs `mid`

| Knob | Paper | **`mid`** |
|------|-------|-----------|
| Task | WMT'14 En→Fr | same |
| $L\times H$ / $E$ | $4\times1000$ / 1000 | **$4\times256$ / 256** |
| $\|V_x\|/\|V_y\|$ | 160k / 80k | **20k / 20k** |
| Train pairs | ~12M | **150k** |
| Max $\|x\|,\|y\|$ | 100 | **50** |
| $B$ | 128 | **64** |
| Epochs | 7.5 | **3** |
| LR | 0.7; hold 5; ×½ / 0.5 ep | 0.7; hold **2**; ×½ / 0.5 ep |
| Grad clip | $\|g\|_2>5$ | same |
| Init | Unif$[-0.08,0.08]$ | same |
| Reverse $x$ | yes | yes |
| Hardware | 8 GPU | **1 GPU** |

**Epochs.** One epoch = one pass over the training pairs. Paper: 7.5 over full WMT. `mid`: 3 over 150k for a short wall clock — same loop, smaller budget.

**Learning rate.** SGD at $0.7$, flat for `lr_hold_epochs`, then $\times\tfrac12$ every 0.5 epoch (§3.4). Paper holds 5 ep; `mid` holds 2 so decay still appears inside 3 epochs.

**Grad clip.** If $\|g\|_2>5$, set $g\leftarrow 5\,g/\|g\|_2$. Stabilises deep LSTMs + large softmax; same threshold as the paper.


In [ ]:
mid, paper = mid_config(), paper_config()
for label, a, b in [
    ("L", paper.num_layers, mid.num_layers),
    ("H", paper.hidden_size, mid.hidden_size),
    ("E", paper.embed_size, mid.embed_size),
    ("|Vx|/|Vy|", f"{paper.src_vocab_size}/{paper.tgt_vocab_size}", f"{mid.src_vocab_size}/{mid.tgt_vocab_size}"),
    ("B", paper.batch_size, mid.batch_size),
    ("epochs", paper.epochs, mid.epochs),
    ("LR hold", paper.lr_hold_epochs, mid.lr_hold_epochs),
]:
    print(f"{label:12} paper={a!s:>14}  mid={b!s:>14}")
print("\nmid LR schedule:")
for ep in [0.0, 1.0, 2.0, 2.5, 3.0]:
    print(f"  epoch {ep:.1f} → {learning_rate_at_epoch(mid, ep):.4g}")


## 2. From plain RNN → LSTM (cell diagrams)

### Vanilla RNN (hidden state only)

At each time $t$: $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$. Information and gradients must travel **only** through $h$.

```
 TIME →
        t=1         t=2         t=3
      ┌──────┐    ┌──────┐    ┌──────┐
 h:   │ h₁   │───►│ h₂   │───►│ h₃   │───► …
      └──▲───┘    └──▲───┘    └──▲───┘
         │           │           │
         x₁          x₂          x₃
```

**Problem:** long-range signal and gradients fade through the repeated $\tanh$ path.

### LSTM (hidden + cell)

Extra highway $c_t$ updated by gates $i,f,o$ (and candidate $g$). $h_t$ is a gated readout of $c_t$.

```
 TIME →
        t=1              t=2              t=3
      ┌──────┐         ┌──────┐         ┌──────┐
 c:   │ c₁   │────────►│ c₂   │────────►│ c₃   │───► …   ← cell highway
      └──┬───┘         └──┬───┘         └──┬───┘
         │ ⊙ tanh         │ ⊙ tanh         │ ⊙ tanh
      ┌──▼───┐         ┌──▼───┐         ┌──▼───┐
 h:   │ h₁   │─(gates)─►│ h₂   │─(gates)─►│ h₃   │───► …
      └──▲───┘         └──▲───┘         └──▲───┘
         │                │                │
         x₁               x₂               x₃
```

Gates (Graves / paper §2):

$$
\begin{aligned}
i_t&=\sigma(W_{xi}x_t+W_{hi}h_{t-1}+b_i),&
f_t&=\sigma(W_{xf}x_t+W_{hf}h_{t-1}+b_f),\\
g_t&=\tanh(W_{xg}x_t+W_{hg}h_{t-1}+b_g),&
o_t&=\sigma(W_{xo}x_t+W_{ho}h_{t-1}+b_o),\\
c_t&=f_t\odot c_{t-1}+i_t\odot g_t,&
h_t&=o_t\odot\tanh(c_t).
\end{aligned}
$$

| Tensor | Shape (`mid`) |
|--------|----------------|
| $x_t$ | $(B,E)$ or $(B,H)$ if upper layer |
| $h_t,c_t$ | $(B,H)=(B,256)$ |
| gate pre-acts | $(B,4H)$ packed $[i\|f\|g\|o]$ |

**Fix vs RNN:** better long-term **information** path via $c$. Still **sequential** in time.


In [ ]:
cell = LSTMCell(input_size=cfg.embed_size, hidden_size=cfg.hidden_size).to(device)
B = 2
x = torch.randn(B, cfg.embed_size, device=device)
h0 = torch.zeros(B, cfg.hidden_size, device=device)
c0 = torch.zeros(B, cfg.hidden_size, device=device)
h1, c1 = cell(x, (h0, c0))
print(f"x {tuple(x.shape)} → h {tuple(h1.shape)}, c {tuple(c1.shape)}")
print(f"params/cell: {sum(p.numel() for p in cell.parameters()):,}")


## 3. Deep encoder unrolled (§2)

Paper stacks $L$ LSTMs. Below: $L=3$, $T=5$ for readability (`mid` uses $L=4$). Vertical = depth, horizontal = time.

**Without cell state** (show $h$ only — same connectivity as an RNN stack):

```
 TIME →
              t=1      t=2      t=3      t=4      t=5
 Layer 3    h³₁ ───► h³₂ ───► h³₃ ───► h³₄ ───►[h³₅] ──► into decoder as part of v
              ▲        ▲        ▲        ▲        ▲
 Layer 2    h²₁ ───► h²₂ ───► h²₃ ───► h²₄ ───► h²₅
              ▲        ▲        ▲        ▲        ▲
 Layer 1    h¹₁ ───► h¹₂ ───► h¹₃ ───► h¹₄ ───► h¹₅
              ▲        ▲        ▲        ▲        ▲
 Input       x₁       x₂       x₃       x₄       x₅
```

Each node $h^\ell_t = \mathrm{LSTM}^\ell(h^{\ell-1}_t,\, h^\ell_{t-1},\, c^\ell_{t-1})$ (with $h^{0}_t\equiv e_t$).

**With cell state** (what we actually run): each layer also carries $c^\ell_t$ along time:

```
 Layer ℓ:   (hℓ₁,cℓ₁) ──► (hℓ₂,cℓ₂) ──► … ──► (hℓ_T,cℓ_T)
                 ▲               ▲                    ▲
            from layer ℓ−1  (or embedding at ℓ=1)
```

**Seq2Seq bottleneck:** the decoder is initialised from the **final** column $t=T$ only — the whole source is compressed into fixed $(h^{1:L}_T,\, c^{1:L}_T)$. Paper emphasises top-layer $v=h^{L}_T$; this repo copies **all** layers’ final $(h,c)$ into the decoder.


In [ ]:
stack = DeepLSTM(cfg.embed_size, cfg.hidden_size, cfg.num_layers).to(device)
B, T = 3, 7
x = torch.randn(B, T, cfg.embed_size, device=device)
lengths = torch.tensor([7, 5, 3], device=device)
out, (h, c) = stack(x, lengths=lengths)
print("out (B,T,H)     ", tuple(out.shape), "  top-layer h_t over time")
print("state h (L,B,H) ", tuple(h.shape), "  = v's hidden part")
print("state c (L,B,H) ", tuple(c.shape))


## 4. Full model dataflow

```
 Source ids x (B,T)
      │ embed  E=256
      ▼
 Encoder DeepLSTM L=4, H=256
      │ final state
      ▼
 v = (h,c) each (L,B,H)  ──────────────────────┐
                                               │ init
 Target ids y_in (B,T')  ─ embed ─► Decoder DeepLSTM L=4
                                               │
                                               ▼
                                    logits (B,T',|Vy|)
                                               │ softmax
                                               ▼
                                    p(y_t | v, y_<t)
```

| Stage | Tensor | Shape (`mid`) |
|-------|--------|----------------|
| `src` | token ids | $(B,T)$ |
| `src_embed` | $e_{1:T}$ | $(B,T,E)$ |
| encoder out | top $h_t$ | $(B,T,H)$ |
| encoder state | $(h,c)$ | $(L,B,H)$ each |
| `tgt_in` | teacher tokens | $(B,T')$ |
| decoder step | $h^{L}_t$ | $(B,H)$ |
| `out_proj` | logits | $(B,T',\|V_y\|)$ |

Loss = mean NLL on non-`<PAD>` target positions.


In [ ]:
Vs, Vt = cfg.src_vocab_size, cfg.tgt_vocab_size
model = Seq2Seq.from_config(cfg, Vs, Vt, pad_id=0).to(device)
model.init_weights(cfg.init_range)
print(f"params @ V={Vs}/{Vt}: {sum(p.numel() for p in model.parameters()):,}")

B, Ts, Tt = 4, 12, 10
src = torch.randint(1, Vs, (B, Ts), device=device)
tgt_in = torch.randint(1, Vt, (B, Tt), device=device)
tgt_out = torch.randint(1, Vt, (B, Tt), device=device)
src_lengths = torch.full((B,), Ts, device=device)
loss, logits = model(src, src_lengths, tgt_in, tgt_out)
print(f"loss={float(loss):.4f}  logits={tuple(logits.shape)}")


## 5. Data pipeline (§3.1, §3.3, §3.4)

1. Load HF `wmt/wmt14` / `fr-en` → En→Fr pairs (cap 150k).
2. Whitespace tokenize (word-level, lowercased).
3. Filter $1\le|x|,|y|\le 50$.
4. Build frequency vocabs ($\le$20k + specials `<PAD><UNK><EOS><SOS>`).
5. **Reverse** source tokens; append `<EOS>`; collate prepends `<SOS>` on the decoder side.
6. Bucket by source length (`bucket_width=5`).

```
EN:  the cat sat on the mat
ENC: mat the on sat cat the <EOS>   →  encoder (time runs on reversed order)
DEC: <SOS> … fr … <EOS>
```

Why reverse (§2, §3.3): shortens the path between early English and early French words in the encoder→decoder chain.


In [ ]:
PEEK_DATA = False  # True → HF download

if PEEK_DATA:
    loader, src_vocab, tgt_vocab, examples = prepare_data(cfg)
    print(f"pairs={len(examples):,}  |Vx|={len(src_vocab)}  |Vy|={len(tgt_vocab)}")
    raw = "the cat sat on the mat"
    print("tokens:", basic_tokenize(raw))
    print("ENC:   ", list(reversed(basic_tokenize(raw))))
    batch = next(iter(loader))
    print({k: tuple(v.shape) for k, v in batch.items()})
else:
    print("Skip download — vocabs come from checkpoint in §6.")


## 6. Train (§3.4) or load checkpoint

SGD, no momentum; clip $\|g\|_2$; LR schedule above; log every 20 steps; sample every 100.
Writes `runs/mid/checkpoint.pt` + `monitor_history.json`.


In [ ]:
DO_TRAIN = False

if DO_TRAIN:
    train(cfg, synthetic=False, sample_src="the cat sat on the mat")

if not CKPT.exists():
    raise FileNotFoundError(f"Missing {CKPT}. Run: python -m seq2seq.train --config mid --device cuda")

ckpt = load_checkpoint(CKPT, device="cuda")
src_vocab, tgt_vocab = ckpt["src_vocab"], ckpt["tgt_vocab"]
model = Seq2Seq.from_config(cfg, len(src_vocab), len(tgt_vocab), tgt_vocab.pad_id)
model.load_state_dict(ckpt["model"])
model.to(device).eval()
print(f"step={ckpt.get('step')}  |Vx|={len(src_vocab)} |Vy|={len(tgt_vocab)}  "
      f"params={sum(p.numel() for p in model.parameters()):,}")


## 7. Training curves


In [ ]:
if HIST.exists():
    history = json.loads(HIST.read_text())
    last = history[-1]
    print(f"{len(history)} rows  final loss={last['loss']:.4f} ppl={last['ppl']:.1f} lr={last['lr']:.4g}")
    if last.get("sample"):
        print(last["sample"])
else:
    print("no monitor_history.json")


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt and HIST.exists():
    history = json.loads(HIST.read_text())
    steps = [r["step"] for r in history]
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
    ax[0].plot(steps, [r["loss"] for r in history]); ax[0].set(title="NLL", xlabel="step")
    ax[1].plot(steps, [r["ppl"] for r in history]); ax[1].set(title="ppl", xlabel="step", yscale="log")
    ax[2].plot(steps, [r["lr"] for r in history]); ax[2].set(title="LR", xlabel="step")
    plt.tight_layout(); plt.show()


## 8. Decode (§3.2)

Encode reversed $x$ → $v$; generate from `<SOS>` with greedy or beam (paper: beam 2 ≈ most of the gain). Softmax still over full $\|V_y\|$.


In [ ]:
for text in [
    "the cat sat on the mat",
    "hello world",
    "I love neural networks",
]:
    src_t, lens = encode_source_sentence(text, src_vocab, reverse=True, device=device)
    g = greedy_decode(model, src_t, lens, tgt_vocab.sos_id, tgt_vocab.eos_id, cfg.max_decode_len)[0]
    b = beam_search_decode(
        model, src_t, lens, tgt_vocab.sos_id, tgt_vocab.eos_id, cfg.max_decode_len, beam_size=2,
    )[0]
    print(f"EN:     {text}")
    print(f"ENC in: {' '.join(reversed(basic_tokenize(text)))}")
    print(f"greedy: {tgt_vocab.decode(g)}")
    print(f"beam2:  {tgt_vocab.decode(b)}\n")


## 9. Where seq2seq sits in the lineage

This workshop is the **Sutskever** box. The rest is context for what came before and after.

```
RNN
│
│ Problem:
│ Information + computation must travel
│ sequentially through time (vanishing gradients).
│
▼
LSTM
│
│ Fix:
│ Cell state = long-term information highway.
│ Still sequential in time.
│
▼
Seq2Seq (Sutskever 2014)   ← you are here
│
│ Idea:
│ Encoder compresses x into fixed v; decoder LM reads v.
│
│ Problem:
│ Entire input → one fixed-dimensional representation.
│ Early tokens must survive the whole encoder path.
│ (Source reversal helps; capacity is still bounded by H.)
│
▼
Attention (Bahdanau 2015)
│
│ Fix:
│ Decoder query can read *all* encoder states h₁…h_T,
│ not only the final column.
│
▼
Self-Attention
│
│ Insight:
│ Positions within one sequence attend to each other
│ (no recurrence required for mixing).
│
▼
Transformer
│
│ Insight:
│ If attention handles communication, drop recurrence.
│ All positions in parallel + positional encodings.
```

| Era | Communication | Parallel over time? |
|-----|---------------|---------------------|
| RNN / LSTM | along $t$ via $h$ (and $c$) | no |
| Seq2Seq | final $v$ only into decoder | no (enc then dec) |
| + Attention | decoder ↔ every encoder $t$ | enc still sequential |
| Transformer | pairwise attention | yes |

### Paper map

| § | Topic | In this notebook |
|---|-------|------------------|
| §2 | $p(y\mid x)$, deep LSTM, reverse | §§2–5 |
| §3.1 | WMT, vocab | §5 |
| §3.2 | beam decode | §8 |
| §3.4 | SGD, clip, buckets, schedule | §§1, 6–7 |
| §4 | BLEU / ensemble | out of scope for `mid` |

### References

1. Sutskever, Vinyals, Le. NeurIPS 2014. [arXiv:1409.3215](https://arxiv.org/abs/1409.3215)
2. Graves. 2013 (LSTM formulation). [arXiv:1308.0850](https://arxiv.org/abs/1308.0850)
3. Bahdanau, Cho, Bengio. 2015 (attention — next step after this paper). [arXiv:1409.0473](https://arxiv.org/abs/1409.0473)
4. Vaswani et al. 2017 (Transformer). [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
